In [ ]:
from common_cl_code import recording_base_path, running_on_real_device
import tables
import datetime
from functools import reduce
from pathlib import Path
import cl
from cl import RecordingView
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA, NMF

rng = np.random.default_rng(0)


In [ ]:
baselines = []

if running_on_real_device:
    for recording in sorted(recording_base_path.glob('*.h5')):
        r = RecordingView(recording)
        s = r.stims.col('channel')
        if len(s) == 0 and r.attributes['duration_seconds'] > 10 * 60 and '2026-04-14' not in recording.name:
            print(recording)
            baselines.append(recording)
else:
    baselines = [
        # "/mnt/data/cl/2026-04-14_20-02-27.846+00-00_recording.h5",
        "/mnt/data/cl/2026-04-24_18-35-30.503+00-00_recording.h5",
        "/mnt/data/cl/2026-04-27_15-58-47.422+00-00_recording.h5",
        "/mnt/data/cl/2026-04-27_21-58-48.318+00-00_recording.h5",
    ]

In [ ]:
for baseline_file in baselines:
    r = RecordingView(baseline_file)
    print(r.spikes.shape[0] / r.attributes['duration_seconds'])


In [ ]:
recording = RecordingView(baselines[2])
recording.plot_spikes_and_stims()

In [ ]:
from common_cl_code.datasets import ArrayWithTime

spikes_per_day = []
days = []
for baseline_file in baselines:
    recording = RecordingView(baseline_file)
    spikes = ArrayWithTime(recording.spikes.col('channel'), recording.spikes.col('timestamp') / recording.attributes['frames_per_second'])
    spikes_per_day.append(spikes)
    days.append(datetime.datetime.fromisoformat(recording.attributes['created_utc']))


In [ ]:
average_frs = []
for spikes in spikes_per_day:
    average_frs.append(np.histogram(spikes, bins=np.arange(0, 64))[0] / (30 * 60))

In [ ]:
fig, ax = plt.subplots()
ax.plot(days, average_frs, '.-')
ax.set_xticks(days)
fig.autofmt_xdate(rotation=45)
ax.set_ylabel('Average firing rate (Hz)')

for day, frs in zip(days, average_frs):
    top3 = np.argsort(frs)[-3:]
    for ch in top3:
        ax.annotate(str(ch), xy=(day, frs[ch]), xytext=(4, 2), textcoords='offset points', fontsize=7)



In [ ]:
reduce(set.intersection, [set(np.where(frs > .01)[0]) for frs in average_frs])

In [ ]:
bin_width = 100 / 1000
a_bins = np.arange(0, 30*60, bin_width)
a_s = []
for spikes in spikes_per_day:
    a = np.array([np.histogram(spikes.t[spikes == channel], bins=a_bins)[0] for channel in range(64)]).T
    a_s.append(a)

nonzero_indices = [set(np.nonzero(a.sum(axis=0) > 30 * 1)[0]) for a in a_s]
nonzero_indices = reduce(set.union, nonzero_indices)

In [ ]:
fig, axs = plt.subplots(ncols=len(baselines), sharey=True, figsize=(12, 4))
covs = []
for a in a_s:
    day: datetime.datetime
    a_sorted = a[:, list(nonzero_indices)]
    cov = np.cov(a_sorted.T)
    off_diagonals = cov - np.diag(np.diag(cov))
    covs.append(np.abs(cov))


vmax = min([np.max(c) for c in covs])
for ax, cov, day in zip(axs, covs, days):
    ax.matshow(np.abs(cov), vmax=vmax, vmin=0)
    ax.set_title(f'{day.date()}')
    ax.set_xticks([])

    ax.set_yticks(range(len(nonzero_indices)))
    ax.set_yticklabels(nonzero_indices, size=6)
axs[0].set_ylabel('Channel')


In [ ]:
def spike_xcorr(spike_times1, spike_times2, bin_edges, same_channel=False):
    """Compute spike-evoked cross-correlogram between two spike trains.

    Returns bin counts (excluding overflow bins) aligned to bin_edges[:-1].
    """
    bins = np.zeros(len(bin_edges) + 1)
    left_bound = 0
    for i, t1 in enumerate(spike_times1):
        for j in range(left_bound, len(spike_times2)):
            if spike_times2[j] < t1 + bin_edges[0]:
                left_bound = j
            elif spike_times2[j] > t1 + bin_edges[-1]:
                break
            else:
                if not (same_channel and i == j):
                    bins[np.searchsorted(bin_edges, t1 - spike_times2[j])] += 1
    return bins[1:-1]


In [ ]:
step = 0.01
bin_edges = np.arange(-.5, .5+step, step)
jitter = 0

fig, axs = plt.subplots(nrows=1, ncols=len(baselines), squeeze=False, constrained_layout=True, figsize=(12,4))

# 24, 42, 57
c1 = 42
c2 = 57

for ax, spikes, day in zip(axs[0], spikes_per_day, days):
    spike_times1 = spikes.t[spikes == c1]
    spike_times2 = spikes.t[spikes == c2]

    if jitter:
        spike_times1 = spike_times1 + np.random.normal(0, jitter, spike_times1.shape)
        spike_times2 = spike_times2 + np.random.normal(0, jitter, spike_times2.shape)

    bins = spike_xcorr(spike_times1, spike_times2, bin_edges, same_channel=(c1 == c2))

    ax.bar(bin_edges[:-1], bins, width=step)

    ax.set_title(f'c{c1} -> c{c2}, {day.date()}')



In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=len(baselines), constrained_layout=True, figsize=(12,4), squeeze=False, sharey=True)
k = np.exp(np.linspace(0, -5, 300))
k = k / np.sum(k)
# k = [1]
smoothed_as = []
for a, ax, day in zip(a_s, axs[0], days):
    smoothed_a = np.apply_along_axis(lambda x: np.convolve(x, k, mode='same'), 0, a)
    smoothed_as.append(smoothed_a)

    ax.plot(np.arange(500) * bin_width, smoothed_a[:500]/bin_width)
    ax.set_xlabel('time (s)')
    ax.set_ylabel('smoothed firing (Hz)')
    ax.set_title(day.date())


In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=len(baselines), constrained_layout=True, figsize=(12,4), squeeze=False)
pcas = []
for smoothed_a, ax in zip(smoothed_as, axs[0]):
    pca = PCA()
    latents = pca.fit_transform(smoothed_a)
    pcas.append(pca)
    ax.plot(latents[:, 0], latents[:, 1])


    ax.set_xlabel('PC 1')
    ax.set_ylabel('PC2')
    ax.set_title(day.date())
